# CNL Mars 2040 — synthetic SAPPHIRE SPEs

**Main: 100-year event. Extreme stress test: 1,000-year event.** SAPPHIRE sets severity, OMNI supplies the historical timing template, and the verified November 16 SHIELDOSE-2 runs supply tissue-target absorbed dose. OLTARIS remains a separate historical reference.

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util

for module, package in {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "pypdf": "pypdf", "bs4": "beautifulsoup4"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
import pandas as pd
from IPython.display import display

# Optional override, for example in Colab:
# os.environ["CNL_PROJECT_ROOT"] = "/content/drive/MyDrive/cnl"
def find_root():
    if os.environ.get("CNL_PROJECT_ROOT"):
        candidates = [Path(os.environ["CNL_PROJECT_ROOT"]).expanduser()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
        candidates = [p for p in candidates if (p / "data/processed/spe_scenarios.csv").is_file()]
        if not candidates and Path("/content").exists():
            from google.colab import drive
            drive.mount("/content/drive")
            candidates = [p.parents[2] for p in Path("/content/drive/MyDrive").rglob("spe_scenarios.csv")
                          if p.parent.name == "processed" and p.parent.parent.name == "data"]
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) != 1 or not (candidates[0] / "data/processed/spe_scenarios.csv").is_file():
        raise FileNotFoundError(f"Set CNL_PROJECT_ROOT to your cnl folder. Candidates: {candidates}")
    return candidates[0]

ROOT = find_root()
spe_hourly = pd.read_csv(ROOT / "data/processed/spe_hourly.csv", parse_dates=["timestamp_utc"])
shielding_dose = pd.read_csv(ROOT / "data/processed/shielding_dose.csv")
spe_scenarios = pd.read_csv(ROOT / "data/processed/spe_scenarios.csv")
print("Project:", ROOT)


Project: /Users/jack/Documents/cnl


## What is available?
OMNIWeb provides **measured hourly flux**. OLTARIS and SHIELDOSE provide **simulated dose quantities**. SAPPHIRE provides **simulated event-integrated spectra**. Source paths, scenario IDs, units and geometry labels remain in the CSVs; they must stay attached when selecting rows.

In [2]:
for name, frame in {"spe_hourly": spe_hourly, "shielding_dose": shielding_dose, "spe_scenarios": spe_scenarios}.items():
    print(f"{name}: {len(frame):,} rows")
    display(frame.head(3))
display(shielding_dose.groupby(["scenario_id", "dose_quantity", "dose_unit", "geometry"], dropna=False).size().rename("rows").reset_index())
print("Hourly rows with missing measurements:", int(spe_hourly.missing_measurement.sum()))


spe_hourly: 264 rows


,timestamp_utc,hours_since_start,proton_flux_gt10_MeV,proton_flux_gt30_MeV,proton_flux_gt60_MeV,missing_measurement,threshold_order_inversion,source,scenario_id,temporal_basis,source_path,flux_unit
0,1972-08-02 00:00:00+00:00,0.0,1.09,1.07,0.86,False,False,OMNIWeb,OMNIWeb_Aug1972,hourly_average,data/raw/omniweb/omni2_5AkMqBOXbn.lst.txt,protons/(cm²·s·sr)
1,1972-08-02 01:00:00+00:00,1.0,1.00,1.07,0.89,False,True,OMNIWeb,OMNIWeb_Aug1972,hourly_average,data/raw/omniweb/omni2_5AkMqBOXbn.lst.txt,protons/(cm²·s·sr)
2,1972-08-02 02:00:00+00:00,2.0,1.04,1.04,0.86,False,False,OMNIWeb,OMNIWeb_Aug1972,hourly_average,data/raw/omniweb/omni2_5AkMqBOXbn.lst.txt,protons/(cm²·s·sr)


shielding_dose: 2,764 rows


,source,tool,scenario_id,dose_quantity,dose_value,dose_unit,geometry,target,aluminum_depth_g_cm2,tissue_depth_g_cm2,...,proton_nuclear_attenuation_included,source_sha256,source_report_path,source_report_sha256,transport_report_path,transport_report_sha256,input_spectrum_sha256,input_link_evidence_path,input_link_evidence_sha256,association_basis
0,OLTARIS,OLTARIS,OLTARIS_Aug1972_King_Al0p4_gcm2,leukemia_dose_equivalent,815122.4,mSv,aluminum_tissue_depth_grid,tissue,0.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OLTARIS,OLTARIS,OLTARIS_Aug1972_King_Al0p4_gcm2,leukemia_dose_equivalent,238774.8,mSv,aluminum_tissue_depth_grid,tissue,0.05,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OLTARIS,OLTARIS,OLTARIS_Aug1972_King_Al0p4_gcm2,leukemia_dose_equivalent,182511.1,mSv,aluminum_tissue_depth_grid,tissue,0.10,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


spe_scenarios: 162 rows


,scenario_id,energy_MeV,spacecraft_integral_protons_cm2,spacecraft_differential_protons_cm2_MeV,model_1p2AU_integral_protons_cm2,model_1p2AU_differential_protons_cm2_MeV,source,tool,shielding,temporal_basis,source_path,mission_average_exposure_factor,segment_1_exposure_factor,mission_total_hours,segment_1_orbit_exposure_hours,segment_1_exposure_hours,return_period_years,model_distance_AU
0,SAPPHIRE-1-in-100,0.10,2.465000e+11,4.805000e+11,2.465000e+11,4.805000e+11,SPENVIS,SAPPHIRE,No magnetic shielding; model at 1.2 AU,event_or_mission_integrated,data/raw/spenvis/SAPPHIRE-1-in-100/solar-proto...,1.0,1.0,72.0,8.0,72.0,100,1.2
1,SAPPHIRE-1-in-100,0.11,2.419000e+11,4.429000e+11,2.419000e+11,4.429000e+11,SPENVIS,SAPPHIRE,No magnetic shielding; model at 1.2 AU,event_or_mission_integrated,data/raw/spenvis/SAPPHIRE-1-in-100/solar-proto...,1.0,1.0,72.0,8.0,72.0,100,1.2
2,SAPPHIRE-1-in-100,0.12,2.376000e+11,4.110000e+11,2.376000e+11,4.110000e+11,SPENVIS,SAPPHIRE,No magnetic shielding; model at 1.2 AU,event_or_mission_integrated,data/raw/spenvis/SAPPHIRE-1-in-100/solar-proto...,1.0,1.0,72.0,8.0,72.0,100,1.2


,scenario_id,dose_quantity,dose_unit,geometry,rows
0,OLTARIS_Aug1972_King_Al0p4_gcm2,absorbed_dose,mGy,aluminum_tissue_depth_grid,441
1,OLTARIS_Aug1972_King_Al0p4_gcm2,effective_dose_equivalent,mSv,user_reported_aluminum_sphere,1
2,OLTARIS_Aug1972_King_Al0p4_gcm2,leukemia_dose_equivalent,mSv,aluminum_tissue_depth_grid,441
3,OLTARIS_Aug1972_King_Al0p4_gcm2,reported_organ_total,mSv,user_reported_aluminum_sphere,27
4,OLTARIS_Aug1972_King_Al0p4_gcm2,solid_cancer_dose_equivalent,mSv,aluminum_tissue_depth_grid,441
5,OLTARIS_Aug1972_King_Al20_gcm2,absorbed_dose,mGy,aluminum_tissue_depth_grid,441
6,OLTARIS_Aug1972_King_Al20_gcm2,effective_dose_equivalent,mSv,user_reported_aluminum_sphere,1
7,OLTARIS_Aug1972_King_Al20_gcm2,leukemia_dose_equivalent,mSv,aluminum_tissue_depth_grid,441
8,OLTARIS_Aug1972_King_Al20_gcm2,reported_organ_total,mSv,user_reported_aluminum_sphere,27
9,OLTARIS_Aug1972_King_Al20_gcm2,solid_cancer_dose_equivalent,mSv,aluminum_tissue_depth_grid,441


Hourly rows with missing measurements: 8


## Available charts

- 01–03: historical OMNI flux, selected shielding results, and SAPPHIRE spectra.
- [04 — Synthetic tissue-target absorbed-dose rate](charts/04_radiation_dose_rate.py): main and extreme events, external and cabin references.
- [05 — Cumulative tissue-target dose](charts/05_cumulative_exposure.py): configurable seven-hour interval, with its accumulated modeled tissue-target dose displayed.
- [06 — Shielding sensitivity](charts/06_shielding_sensitivity.py): both SAPPHIRE scenarios at all six simulated shielding levels.
- 07: separate clinical reference, not a health classification of mission results.
- 08: synthetic hourly proton fluence.

Each figure has a `.txt` caption. The older King-only figures remain under `outputs/charts/historical/`.

In [3]:
CHART = 8
RUN_CHART = False
if RUN_CHART:
    from IPython.display import Image
    scripts = {1: "01_spe_proton_flux.py", 2: "02_shielding_vs_dose.py", 3: "03_sapphire_spectra.py",
               4: "04_radiation_dose_rate.py", 5: "05_cumulative_exposure.py", 6: "06_shielding_sensitivity.py",
               7: "07_absorbed_dose_health_reference.py", 8: "08_synthetic_sapphire_proton_profile.py"}
    completed = subprocess.run([sys.executable, str(ROOT / "charts" / scripts[CHART])], capture_output=True, text=True)
    print(completed.stdout.strip())
    if completed.returncode:
        print(completed.stderr.strip())
    else:
        display(Image(filename=completed.stdout.strip().splitlines()[-1]))

## Model-Ready Data

The verified SAPPHIRE 100-year and 1,000-year SHIELDOSE-2 event doses are distributed over the **complete assumed 264-hour event** using normalized OMNI timing. The default 72-hour window starts at template hour 48. Eight missing observations are filled only in modeled copies; nine timing alternatives retain the existing gap methods and energy-channel proxies. Shading describes sensitivity, not confidence intervals.

Both simulations use tissue at the centre of aluminium spheres, 4π irradiation, and no proton nuclear attenuation. All six depths (0.4, 1, 2, 5, 10 and 20 g/cm²) are retained. Source rad values are converted using **1 rad = 10 mGy**. These are tissue-target calculations, not astronaut whole-body doses. The spectrum is assumed fixed through time; historical timing does not set event magnitude.

The new source reports identify the return periods and match the existing spectra. The dose tables match their numerical exports at report precision; run pairing and event-integrated interpretation were confirmed by the user. Old ambiguous SHIELDOSE and failed MULASSIS outputs remain excluded. No King dose normalizes these synthetic events.

Change `radiation_config.json` for the window and interval. Select one scenario, location and timing member before integration. The same baseline will serve all strategies; scheduling is not implemented.

**Quantity interpretation:** tissue-target dose is energy deposited per unit mass in an idealized target. Organ dose requires an organ-specific/anatomical model. Whole-body exposure requires a defined anatomical exposure quantity. Effective dose equivalent is in Sv and is not absorbed dose in Gy. The clinical references in Chart 07 cannot directly classify these target doses or validate EVA/cabin hardware.

In [4]:
import json
subprocess.run([sys.executable, str(ROOT / "radiation_model.py")], check=True)
audit = json.loads((ROOT / "outputs/model/sapphire_transport_audit.json").read_text())
print("Synthetic dose status:", audit["status"])
display(pd.DataFrame(audit["scenarios"]))
for message in audit["missing_matches"]:
    print("MISSING:", message)
temporal_weights = pd.read_csv(ROOT / "outputs/model/synthetic_sapphire_temporal_weights.csv")
display(temporal_weights.groupby("member_id").normalized_bin_weight.sum().rename("full_event_weight_sum"))
mission_path = ROOT / "data/processed/mission_radiation_environment.csv"
mission_radiation_environment = pd.read_csv(mission_path) if mission_path.exists() else None
if mission_radiation_environment is None:
    print("No synthetic absorbed-dose CSV: verified transport inputs are required.")
else:
    display(mission_radiation_environment.head())
if mission_radiation_environment is not None:
    central = mission_radiation_environment.query("estimate_type == 'central'")
    display(central.groupby(['scenario', 'shielding_g_cm2']).agg(
        event_total_mGy=('event_absorbed_dose_mGy', 'first'),
        full_event_total_mGy=('cumulative_end_dose_mGy', 'max')))
    interval = pd.read_csv(ROOT / "outputs/model/synthetic_sapphire_interval_exposure.csv")
    display(interval.query("estimate_type == 'central' and location in ['external_eva_reference', 'spacecraft_cabin_reference']"))

# Query storm-relative hours directly from the shared CSV.
from radiation_model import baseline_interval_dose
print("Storm hours 16–23, 0.4 g/cm², main scenario (tissue-target mGy):",
      baseline_interval_dose("SAPPHIRE-1-in-100", 0.4, 16, 23))


Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)


Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)


Ignoring wrong pointing object 17 0 (offset 0)


Synthetic SAPPHIRE spectra and full-record timing verified. Dose status: ready
Synthetic dose status: ready


,scenario_id,return_period_years,spectrum_sha256,spectrum_verified,report_display_distance_AU
0,SAPPHIRE-1-in-100,100,2dcb23e6f02932c5a2fc340be99c6bb0a94d432181f3ed...,True,1.2
1,SAPPHIRE-1-in-1000,1000,6b30e922642a0a8de723f614fce8df8cd97ec889d6dd9c...,True,1.2


member_id
gt10MeV__linear        1.0
gt10MeV__log_linear    1.0
gt10MeV__pchip         1.0
gt30MeV__linear        1.0
gt30MeV__log_linear    1.0
gt30MeV__pchip         1.0
gt60MeV__linear        1.0
gt60MeV__log_linear    1.0
gt60MeV__pchip         1.0
Name: full_event_weight_sum, dtype: float64

,scenario,hour,shielding_g_cm2,dose_rate_mGy_h,cumulative_dose_mGy,timing_variant,data_quality,hour_end,bin_duration_h,cumulative_end_dose_mGy,...,transport_report_sha256,input_link_evidence_path,input_link_evidence_sha256,baseline_version,dose_rate_mGy_h_sensitivity_lower,dose_rate_mGy_h_sensitivity_upper,cumulative_dose_mGy_sensitivity_lower,cumulative_dose_mGy_sensitivity_upper,cumulative_end_dose_mGy_sensitivity_lower,cumulative_end_dose_mGy_sensitivity_upper
0,SAPPHIRE-1-in-100,0.0,0.4,0.166408,0.000000,gt10MeV__linear,observed_template_hour,1.0,1.0,0.166408,...,7aca7bd57be4c1aedea3be4f3828a97b080f74291564c2...,data/raw/spenvis/verified_sapphire_manifest.json,cbf1f1b267c8e782990c54f15f3daeca9ed3997ee58b12...,1.0,0.166383,1.160536,0.000000,0.000000,0.166383,1.160536
1,SAPPHIRE-1-in-100,1.0,0.4,0.152668,0.166408,gt10MeV__linear,observed_template_hour,2.0,1.0,0.319075,...,7aca7bd57be4c1aedea3be4f3828a97b080f74291564c2...,data/raw/spenvis/verified_sapphire_manifest.json,cbf1f1b267c8e782990c54f15f3daeca9ed3997ee58b12...,1.0,0.152645,1.201019,0.166383,1.160536,0.319027,2.361555
2,SAPPHIRE-1-in-100,2.0,0.4,0.158774,0.319075,gt10MeV__linear,observed_template_hour,3.0,1.0,0.477850,...,7aca7bd57be4c1aedea3be4f3828a97b080f74291564c2...,data/raw/spenvis/verified_sapphire_manifest.json,cbf1f1b267c8e782990c54f15f3daeca9ed3997ee58b12...,1.0,0.158750,1.160536,0.319027,2.361555,0.477778,3.522091
3,SAPPHIRE-1-in-100,3.0,0.4,0.137401,0.477850,gt10MeV__linear,observed_template_hour,4.0,1.0,0.615250,...,7aca7bd57be4c1aedea3be4f3828a97b080f74291564c2...,data/raw/spenvis/verified_sapphire_manifest.json,cbf1f1b267c8e782990c54f15f3daeca9ed3997ee58b12...,1.0,0.137380,1.187525,0.477778,3.522091,0.615158,4.709616
4,SAPPHIRE-1-in-100,4.0,0.4,0.155721,0.615250,gt10MeV__linear,observed_template_hour,5.0,1.0,0.770971,...,7aca7bd57be4c1aedea3be4f3828a97b080f74291564c2...,data/raw/spenvis/verified_sapphire_manifest.json,cbf1f1b267c8e782990c54f15f3daeca9ed3997ee58b12...,1.0,0.155698,1.228009,0.615158,4.709616,0.770855,5.937624


event_total_mGy  full_event_total_mGy
scenario           shielding_g_cm2                                       
SAPPHIRE-1-in-100  0.4                     74571.00              74571.00
                   1.0                     22306.00              22306.00
                   2.0                      7963.50               7963.50
                   5.0                      1664.00               1664.00
                   10.0                      476.16                476.16
                   20.0                      134.00                134.00
SAPPHIRE-1-in-1000 0.4                    127280.00             127280.00
                   1.0                     38573.00              38573.00
                   2.0                     13862.00              13862.00
                   5.0                      2903.70               2903.70
                   10.0                      830.93                830.93
                   20.0                      233.83                233.83

,scenario_id,location,member_id,interval_start_hour,interval_end_hour,absorbed_dose_mGy,shielding_g_cm2,estimate_type,scenario_role,dose_quantity,target_type,geometry,target_geometry,unit,nuclear_attenuation
1,SAPPHIRE-1-in-100,external_eva_reference,gt10MeV__log_linear,0,7,575.928908,0.4,central,main,absorbed_dose,idealized_tissue,center_of_aluminum_sphere,centre_of_aluminium_spheres,mGy,False
46,SAPPHIRE-1-in-100,spacecraft_cabin_reference,gt10MeV__log_linear,0,7,1.034913,20.0,central,main,absorbed_dose,idealized_tissue,center_of_aluminum_sphere,centre_of_aluminium_spheres,mGy,False
55,SAPPHIRE-1-in-1000,external_eva_reference,gt10MeV__log_linear,0,7,983.012584,0.4,central,worst_case,absorbed_dose,idealized_tissue,center_of_aluminum_sphere,centre_of_aluminium_spheres,mGy,False
100,SAPPHIRE-1-in-1000,spacecraft_cabin_reference,gt10MeV__log_linear,0,7,1.805923,20.0,central,worst_case,absorbed_dose,idealized_tissue,center_of_aluminum_sphere,centre_of_aluminium_spheres,mGy,False


Storm hours 16–23, 0.4 g/cm², main scenario (tissue-target mGy): 3.8272391527327745
